In [9]:
!pip install -q transformers datasets evaluate sacrebleu rouge_score 
!pip install -q transformers[torch]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [3]:
from transformers import (AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer,
                          DataCollatorForLanguageModeling)
from datasets import Dataset, DatasetDict
import evaluate
import torch
from argparse import Namespace

In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

#
# Data Preprocessing:
# 1. Process <question, answer> pairs to the message text based on the chat format of the model:
# https://huggingface.co/LiquidAI/LFM2-1.2B
#
# [
#    { "role": "system", "content": "You are a helpful assistant trained by Liquid AI."},
#    { "role": "user", "content": "What is C. elegans?"},
#    { "role": "assistant", "content": "It's a tiny nematode that lives in temperate soil environments."}
# ]
# 2. Tokenize the text
#
def prepare_dataset(args, tokenizer):
    raw_dataset = Dataset.from_csv(args.dataset_path).train_test_split(test_size=0.3)
    validation_dataset = raw_dataset["test"].train_test_split(test_size=0.3)
    raw_dataset = DatasetDict({
        "train": raw_dataset["train"],
        "validation": validation_dataset["train"],
        "test": validation_dataset["test"]
    })
    print(raw_dataset)

    def tokenize(batch, tokenizer):
        input_ids, attention_mask = [], []

        for question, answer in zip(batch['question'], batch['answer']):
            message = [
                {
                    "role": "system", "content": "You are a helpful medical diseases assistant"
                },
                {
                    "role": "user", "content": question
                },
                {
                    "role": "assistant", "content": answer
                }
            ]
            text = tokenizer.apply_chat_template(message, tokenize=False)
            text_tokenized = tokenizer(text)

            if args.max_text_length:
                text_tokenized = {k: v[:args.max_text_length] for k, v in text_tokenized.items()}

            input_ids.append(text_tokenized["input_ids"])
            attention_mask.append(text_tokenized["attention_mask"])

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask
        }

    tokenized_dataset = raw_dataset.map(tokenize, fn_kwargs={"tokenizer": tokenizer}, batched=True,
                                        remove_columns=['question', 'answer'])
    return raw_dataset, tokenized_dataset

#
# Model Training:
#  Tune the pre-trained model
#
def train(args, model, tokenizer, dataset):

    acc_metric = evaluate.load("accuracy")

    def preprocess_logits(logits, labels):
        if isinstance(logits, tuple):
            logits = logits[0]

        return logits.argmax(dim=-1)

    def compute_metrics(eval_preds):
        preds, labels = eval_preds

        labels = labels[:, 1:].reshape(-1)
        preds = preds[:, :-1].reshape(-1)
        return acc_metric.compute(predictions=preds, references=labels)

    args = TrainingArguments(
        output_dir=args.output_dir,
        per_device_train_batch_size=args.per_device_train_batch_size,
        per_device_eval_batch_size=args.per_device_eval_batch_size,
        learning_rate=args.learning_rate,
        weight_decay=args.weight_decay,
        # max_steps=20,
        lr_scheduler_type="cosine",
        warmup_steps=5,
        # log_level="info",
        # logging_steps=1,
        save_total_limit=1,
        seed=42,
        bf16=args.bf16,
        eval_strategy=args.eval_strategy,
        eval_steps=args.eval_steps,
        gradient_checkpointing=True,
        report_to="none"
    )
    trainer = Trainer(
        model=model,
        args=args,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, return_tensors='pt'),
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
        preprocess_logits_for_metrics=preprocess_logits
    )
    trainer.train()

#### 1. set training configuration
####     - choose pre-trained model LiquidAI/LFM2-1.2B as base model because it's relative small, but with good performance benchmark.
####     - set small batch size / learning rate since the training is run with constrained computing resources. 

In [5]:
args = Namespace(
    model_name="LiquidAI/LFM2-1.2B",
    dataset_path="data/mle_screening_dataset.csv",
    output_dir="mle_screening_lfm2_1pt2b",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    learning_rate=5e-5,
    weight_decay=0.01,
    bf16=True if torch.cuda.is_available() else False,
    torch_dtype="bfloat16" if torch.cuda.is_available() else "int32",
    eval_strategy="epoch",
    eval_steps=10,
    max_text_length=2048
)

#### 2. Load the tokenizer & model:

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(args.model_name)

model = AutoModelForCausalLM.from_pretrained(args.model_name, torch_dtype="bfloat16")

#### 3. Process the raw data to training data

In [7]:
raw_dataset, tokenized_dataset = prepare_dataset(args, tokenizer)

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 11484
    })
    validation: Dataset({
        features: ['question', 'answer'],
        num_rows: 3445
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1477
    })
})


Map:   0%|          | 0/11484 [00:00<?, ? examples/s]

Map:   0%|          | 0/3445 [00:00<?, ? examples/s]

Map:   0%|          | 0/1477 [00:00<?, ? examples/s]

#### 4. Train the model

In [8]:
train(args, model, tokenizer, tokenized_dataset)

/tmp/ipykernel_338/3333384389.py:98: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.966700,0.936314,0.965004
2,0.628400,0.949510,0.965539
3,0.524700,0.983221,0.965201


In [9]:
#
# Example interactions
#
def generate_prediction(model, tokenizer, questions):
    messages = []
    for question in questions:
        messages.append([
            {"role": "system", "content": "You are a helpful medical diseases assistant"
             },
            {"role": "user", "content": question}])

    model_inputs = tokenizer.apply_chat_template(messages,
                                                 tokenize=True, add_generation_prompt=True, return_tensors="pt",
                                                 padding=True).to(model.device)
    model_generations = model.generate(model_inputs, do_sample=True, temperature=0.3, min_p=0.15,
                                       repetition_penalty=1.05, max_new_tokens=1000)
    model_generations_texts = tokenizer.batch_decode(model_generations, skip_special_tokens=True)
    predictions = []
    for generation in model_generations_texts:
        if "\nassistant\n" not in generation:
            predictions.append(" ")
        else:
            resp = generation.split("\nassistant\n")[1]
            predictions.append(resp)

    return predictions

#
# Evaluate model:
#  bleu & rogue score
#
def eval_model(model, tokenizer, test_dataset):
    bleu_metric = evaluate.load("sacrebleu")
    rouge_metric = evaluate.load("rouge")

    tokenizer.padding_side = "left"

    references = test_dataset["answer"]
    predictions = generate_prediction(model, tokenizer, test_dataset["question"])
    bleu_score = bleu_metric.compute(references=references, predictions=predictions)
    rouge_score = rouge_metric.compute(references=references, predictions=predictions)

    return {
        "bleu_score": bleu_score,
        "rouge_score": rouge_score
    }

#### 5. Evaluate the model
####    - bleu 
####    - rogue

In [12]:
scores = eval_model(model, tokenizer, raw_dataset["test"].select(range(int(len(raw_dataset["test"]) * 0.2))))
print(scores)

{'bleu_score': {'score': 22.98621748813098, 'counts': [23817, 15594, 12835, 11611], 'totals': [43968, 43673, 43378, 43083], 'precisions': [54.16894104803494, 35.706271609461226, 29.588731615104432, 26.950305224798644], 'bp': 0.6522608166743934, 'sys_len': 43968, 'ref_len': 62756}, 'rouge_score': {'rouge1': np.float64(0.4505135040408538), 'rouge2': np.float64(0.29154321271594685), 'rougeL': np.float64(0.36178977946270346), 'rougeLsum': np.float64(0.36901375344369536)}}


#### 6. Sample interactions

In [13]:
sample_questions = [
    "Who is at risk for Diabetes?",
    "What is (are) Dry Mouth",
    "What is (are) Stroke?"
]
predictions = generate_prediction(model, tokenizer, sample_questions)

In [18]:
for i, (q, a) in enumerate(zip(sample_questions, predictions)):
    print(f"Question {i+1}:\n", q)
    print("Answer:\n", a)
    print("-" * 10)

Question 1:
 Who is at risk for Diabetes?
Answer:
 Diabetes affects people of all ages, races, and sexes. However, certain factors increase the risk of developing diabetes.
                
Age. The risk of developing diabetes increases with age. Most people who develop diabetes are over 45.
                
Race. People of Hispanic or Latino ancestry are more likely to develop diabetes than any other group. Asian Americans also have a higher risk of developing diabetes than other groups. African Americans and American Indians tend to develop diabetes at a younger age than other groups, but they are more likely to develop serious health problems related to diabetes.
                
Sex. Men are more likely to develop diabetes than women. However, as more women develop diabetes, the gap between men's and women's diabetes rates begins to close.
                
Family History. If you have a parent, brother, or sister who has diabetes, you are at higher risk of developing the disease. Ha